In [ ]:
import torch 
batch = 64 
seq = 12 
d = 500 
q = torch.rand((batch, seq, d ))
k = torch.rand((batch, seq, d ))
v = torch.rand((batch , seq, d ))
# so we tile the q and k and v loads from HBM to SRAM ( onchip mem) and then do the onlien softmax fro this 
normal_softmax = q@k.transpose(-1, -2 ) / torch.sqrt(torch.tensor(d, dtype=torch.float32))
normal_softmax = torch.softmax(normal_softmax, dim = -1 ) @v
normal_softmax.shape # batch x seq_len x seq_len so this takes O(seq2 x d )

In [ ]:
i = 0 
l = 0 
import math 

score_i = q[i] @ k[i].transpose(-1, -2 ) / math.sqrt(d)  #this is still per rpw at a tiem, we need to make this into per k columns at a time 

m = torch.max(score_i)
l += torch.exp(score_i - m ) /  sum(torch.exp(score_i - m ))

m.shape 
# then the correction for m can go here and we need to run the @v after that 

attn_block = l @ v[i]
l.shape

In [ ]:
import math
import torch 
batch = 64 
seq = 12 
d = 500 
# q = torch.rand((batch, seq, d ))
# k = torch.rand((batch, seq, d ))
# v = torch.rand((batch , seq, d ))
block_size = 4 
O = torch.zeros(batch, seq, d)
res = []
for b in range(batch):
    for i in range(seq):
        m = float("-inf")
        l = 0.0
        o = torch.zeros(d)



        for j in range(0, seq, block_size): 

            score_i = q[b,i] @ k[b, j:j+block_size].transpose(-1, -2 ) / math.sqrt(d)  #this is still per rpw at a tiem, we need to make this into per k columns at a time 

            m_new = max(m , torch.max(score_i)) 
            l = l* torch.exp(m  - m_new ) 
            o = o* math.exp(m - m_new)
            # this is the correction for the output  and running  max 

            exp_s = torch.exp(score_i - m_new)
            l += exp_s.sum()
            o += exp_s @ v[b,  j:j+block_size ]
            m = m_new 
        o = o /l 
        O[b, i] = o 
O.shape

In [58]:
print((O - normal_softmax).abs().max())


tensor(8.3447e-07)


In [ ]:
# flash attention si very useful when we are mem bound in longer seq and not compute boudn 
# this is why this is great fro long context task because attention scales with seq**2 
# flash attention does not help oin the case of Short sequences (N < 256) — compute-bound, not memory-bound
# Very large head dim — tile doesn't fit in SRAM
# Non-standard attention patterns (sparse, linear) — tiling assumptions break


---

## Interview Questions I Got Wrong / Need to Review

### Q3: Why is recomputing the attention matrix in backward pass a net win?

**My wrong instinct:** "it's the same computation just in a running way"

**Correct answer:** Standard attention saves the full N×N matrix during forward for backprop → O(N²) memory. Flash attention throws it away and only saves `m` and `l` per row (O(N) total). During backward, it recomputes each tile on-the-fly from Q, K, V. This is a net win because:
- We're **memory-bound**, not compute-bound
- The extra FLOPs are cheap (matmul in SRAM, GPU was idle anyway)
- Freeing O(N²) memory lets us run longer sequences / bigger batches

### Q5: What block size to choose given SRAM of size M?

**My wrong answer:** N×N×d / M should be integer

**Correct answer:** One tile must fit entirely in SRAM simultaneously:
- Q block: `B_q × d`
- K block: `B_kv × d`  
- V block: `B_kv × d`
- Score tile: `B_q × B_kv`
- Output block: `B_q × d`

Constraint: `(B_q × d + 2 × B_kv × d + B_q × B_kv + B_q × d) × bytes_per_float ≤ M`

Larger blocks = fewer HBM round-trips = faster, but capped by SRAM capacity.
In practice (A100): M ≈ 20MB, d = 128, fp16 → block sizes of 64-128.

### Q4: Why does causal masking give ~2× additional speedup?

**My partial answer:** "we mask out future tokens"

**Complete answer:** The upper triangle of the N×N matrix is all masked (-inf). That's ~half the tiles. Flash attention can **skip entire KV blocks** where all key positions are in the future — never loads them, never computes them. Standard attention doesn't get this because it materializes the full matrix first, then masks. Flash checks before loading → ~50% of tiles skipped → ~2× speedup.

---

## Key Concepts Summary

| | Standard Attention | Flash Attention |
|---|---|---|
| HBM memory | O(N²) for attn matrix | O(N) output only |
| HBM IO | O(N² + Nd) | O(N²d / M) |
| FLOPs | O(N²d) | O(N²d) — same |
| Backward storage | Full N×N matrix | Just m, l per row |
| Causal speedup | None (full matrix first) | ~2× (skip future tiles) |

## When Flash Attention Does NOT Help
- Short sequences (N < 256) — compute-bound, not memory-bound
- Very large head dim — tile doesn't fit in SRAM
- Non-standard attention patterns (sparse, linear) — tiling assumptions break